In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/test_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/val_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation dataset/train_80_mode_no_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/test_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/val_10_mode.xlsx
/kaggle/input/datasets/fairoooz/sagorer/mode transformation augmentation dataset/train_80_mode_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/train_80_missing_no_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/val_10.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation dataset/test_10.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/train_80_missing_smote.xlsx
/kaggle/input/datasets/fairoooz/sagorer/missing t

In [2]:
# # ============================================================
# # CARDIAC MI STAGING — BioMedBERT (Text-ified Tabular)
# # TARGET: Phase (Non-MI, Chronic, Sub-acute, Acute)
# # ============================================================
# import subprocess, sys, os, warnings
# warnings.filterwarnings("ignore")

# def pip_install(pkg):
#     cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]
#     result = subprocess.run(cmd, capture_output=True, text=True)
#     if result.returncode == 0:
#         print(f"[OK] {pkg}")
#     else:
#         print(f"[WARN] {pkg}: {result.stderr[-200:]}")

# pip_install("transformers")
# pip_install("datasets")
# pip_install("accelerate")
# pip_install("evaluate")

# import numpy as np
# import pandas as pd
# import torch
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import (
#     accuracy_score, precision_score, recall_score, f1_score,
#     classification_report, confusion_matrix
# )
# from transformers import (
#     AutoTokenizer, 
#     AutoModelForSequenceClassification, 
#     Trainer, 
#     TrainingArguments, 
#     DataCollatorWithPadding
# )
# from datasets import Dataset

# # ====================== CONFIG ======================
# TRAIN_PATH = "/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/train_80_missing_smote.xlsx"
# TEST_PATH  = "/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/test_10.xlsx"
# TARGET_COL = "Phase"
# RANDOM_SEED = 42

# # ── Top-10 features from feature importance chart ──────────────────────────
# SELECTED_FEATURES = [
#     "Blood_Glucose_mmolL",
#     "Diastolic_BP",
#     "Chest_Pain",
#     "Weighted_Risk_Score",
#     "Systolic_BP",
#     "Heart_Rate_bpm",
#     "Weighted_Symptom_Score",
#     "Serum_Creatinine_mgdL",
#     "hs_Troponin_I_ngL",
#     "Age_Adjusted_Troponin",
# ]
# # ───────────────────────────────────────────────────────────────────────────

# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# torch.manual_seed(RANDOM_SEED)
# np.random.seed(RANDOM_SEED)

# print(f"Device: {DEVICE}")
# print(f"Selected features ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}")

# # ============================================================
# # DATA LOADING & TEXT SERIALIZATION
# # ============================================================
# def load_and_serialize(train_path, test_path):
#     train_df = pd.read_excel(train_path)
#     test_df  = pd.read_excel(test_path)

#     # ── Auto-rename target if missing in test set ──────────────────────────
#     if TARGET_COL not in test_df.columns:
#         for col in test_df.columns:
#             if test_df[col].astype(str).isin(
#                 train_df[TARGET_COL].astype(str).unique()
#             ).sum() > 5:
#                 test_df = test_df.rename(columns={col: TARGET_COL})
#                 print(f"[INFO] Auto-renamed target to '{TARGET_COL}'")
#                 break

#     print(f"Train shape (raw): {train_df.shape} | Test shape (raw): {test_df.shape}")

#     # ── Validate that all selected features exist ──────────────────────────
#     missing_train = [f for f in SELECTED_FEATURES if f not in train_df.columns]
#     missing_test  = [f for f in SELECTED_FEATURES if f not in test_df.columns]
#     if missing_train:
#         raise ValueError(f"Missing in train: {missing_train}")
#     if missing_test:
#         raise ValueError(f"Missing in test:  {missing_test}")

#     # ── Keep only selected features + target ──────────────────────────────
#     train_df = train_df[SELECTED_FEATURES + [TARGET_COL]]
#     test_df  = test_df[SELECTED_FEATURES  + [TARGET_COL]]

#     print(f"Train shape (filtered): {train_df.shape} | Test shape (filtered): {test_df.shape}")
#     print(f"Train Phase distribution:\n{train_df[TARGET_COL].value_counts()}\n")

#     # ── Encode labels ──────────────────────────────────────────────────────
#     le = LabelEncoder()
#     le.fit(train_df[TARGET_COL])
#     y_train = le.transform(train_df[TARGET_COL])
#     y_test  = le.transform(test_df[TARGET_COL])

#     global CLASS_NAMES
#     CLASS_NAMES = le.classes_.tolist()
#     print(f"Classes: {CLASS_NAMES}")

#     # ── Features only (target already encoded) ────────────────────────────
#     X_train = train_df[SELECTED_FEATURES]
#     X_test  = test_df[SELECTED_FEATURES]

#     # ── Convert tabular row → natural text ────────────────────────────────
#     def row_to_text(row):
#         return " | ".join(
#             [f"{col}:{val}" for col, val in row.items() if pd.notna(val)]
#         )

#     train_texts = X_train.apply(row_to_text, axis=1).tolist()
#     test_texts  = X_test.apply(row_to_text, axis=1).tolist()

#     # Preview
#     print(f"\nSample serialized text:\n  {train_texts[0]}\n")

#     return train_texts, test_texts, y_train, y_test, CLASS_NAMES


# train_texts, test_texts, y_train, y_test, CLASS_NAMES = load_and_serialize(
#     TRAIN_PATH, TEST_PATH
# )

# # ============================================================
# # BioMedBERT SETUP
# # ============================================================
# print("\n" + "█"*70)
# print(" MODEL: BioMedBERT")
# print("█"*70)

# MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# # ── Hugging Face Datasets ─────────────────────────────────────────────────
# train_data = Dataset.from_dict({"text": train_texts, "label": y_train.tolist()})
# test_data  = Dataset.from_dict({"text": test_texts,  "label": y_test.tolist()})

# def tokenize_function(examples):
#     return tokenizer(
#         examples["text"],
#         padding="max_length",
#         truncation=True,
#         max_length=256,   # 10 features → much shorter texts; 256 is sufficient
#     )

# train_data = train_data.map(tokenize_function, batched=True)
# test_data  = test_data.map(tokenize_function, batched=True)

# train_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])
# test_data.set_format("torch",  columns=["input_ids", "attention_mask", "label"])

# # ── Model ─────────────────────────────────────────────────────────────────
# num_labels = len(CLASS_NAMES)
# model = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_NAME,
#     num_labels=num_labels,
#     ignore_mismatched_sizes=True,
# ).to(DEVICE)

# # ====================== TRAINING ARGS ======================
# training_args = TrainingArguments(
#     output_dir="./biomedbert_results",
#     num_train_epochs=5,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=16,
#     warmup_steps=50,
#     weight_decay=0.01,
#     logging_steps=10,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="f1",
#     report_to="none",
#     seed=RANDOM_SEED,
#     remove_unused_columns=False,
# )

# # ====================== METRICS ======================
# def compute_metrics(eval_pred):
#     predictions, labels = eval_pred
#     preds = np.argmax(predictions, axis=1)
#     return {
#         "accuracy":  accuracy_score(labels, preds),
#         "f1":        f1_score(labels, preds, average="weighted", zero_division=0),
#         "precision": precision_score(labels, preds, average="weighted", zero_division=0),
#         "recall":    recall_score(labels, preds, average="weighted", zero_division=0),
#     }

# # ====================== TRAINER ======================
# data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_data,
#     eval_dataset=test_data,
#     compute_metrics=compute_metrics,
#     data_collator=data_collator,
# )

# # ====================== TRAINING ======================
# print("Starting BioMedBERT fine-tuning on selected features...")
# trainer.train()

# # ====================== EVALUATION ======================
# print("\nEvaluating on test set...")
# eval_results = trainer.evaluate()
# print("Evaluation Results:", eval_results)

# # ── Final Predictions ─────────────────────────────────────────────────────
# predictions = trainer.predict(test_data)
# preds = np.argmax(predictions.predictions, axis=1)

# print("\n" + "="*70)
# print("BioMedBERT FINAL RESULTS  (top-10 features only)")
# print("="*70)
# print(classification_report(y_test, preds, target_names=CLASS_NAMES, zero_division=0))

# cm = confusion_matrix(y_test, preds)
# print("\nConfusion Matrix:\n", cm)

# print("\n✅ BioMedBERT training & evaluation completed!")

In [3]:
# ============================================================
# CARDIAC MI STAGING — BioMedBERT: ALL features vs SELECTED (top-10)
# Goal: show whether feature selection is what limits the transformer.
# Everything except the feature set is identical between the two runs
# (same model, same hyperparameters, same seed(s), same eval protocol as
# your original BioMedBERT script). Reports metrics + training time.
# ============================================================
import subprocess, sys, os, time, gc, warnings
warnings.filterwarnings("ignore")

def pip_install(pkg):
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(f"[OK] {pkg}" if result.returncode == 0 else f"[WARN] {pkg}: {result.stderr[-200:]}")

for p in ["transformers", "datasets", "accelerate", "evaluate"]:
    pip_install(p)

import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding, set_seed
)
from datasets import Dataset

# ====================== CONFIG ======================
TRAIN_PATH = "/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/train_80_missing_smote.xlsx"
TEST_PATH  = "/kaggle/input/datasets/fairoooz/sagorer/missing transformation augmentation dataset/test_10.xlsx"
TARGET_COL = "Phase"

# Use several seeds if you want the comparison to be robust to run-to-run noise
# (e.g. [42, 43, 44]). Each seed multiplies total runtime.
SEEDS = [42]

SELECTED_FEATURES = [
    "Blood_Glucose_mmolL", "Diastolic_BP", "Chest_Pain", "Weighted_Risk_Score",
    "Systolic_BP", "Heart_Rate_bpm", "Weighted_Symptom_Score",
    "Serum_Creatinine_mgdL", "hs_Troponin_I_ngL", "Age_Adjusted_Troponin",
]

# Columns to leave out of the "all features" set (IDs, row numbers, leakage columns, ...)
EXCLUDE_COLS = []

MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
BERT_MAX_LEN = 512   # hard limit of BERT positional embeddings

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ============================================================
# DATA LOADING
# ============================================================
train_df = pd.read_excel(TRAIN_PATH)
test_df  = pd.read_excel(TEST_PATH)

if TARGET_COL not in test_df.columns:
    for col in test_df.columns:
        if test_df[col].astype(str).isin(train_df[TARGET_COL].astype(str).unique()).sum() > 5:
            test_df = test_df.rename(columns={col: TARGET_COL})
            print(f"[INFO] Auto-renamed target to '{TARGET_COL}'")
            break

# "All features" = every column present in BOTH files, except target/excluded
ALL_FEATURES = [c for c in train_df.columns
                if c != TARGET_COL and c in test_df.columns and c not in EXCLUDE_COLS]

for f in SELECTED_FEATURES:
    assert f in ALL_FEATURES, f"Selected feature '{f}' not found in both train/test"

print(f"Train: {train_df.shape} | Test: {test_df.shape}")
print(f"All features: {len(ALL_FEATURES)} | Selected features: {len(SELECTED_FEATURES)}")
print(f"Train Phase distribution:\n{train_df[TARGET_COL].value_counts()}\n")

le = LabelEncoder().fit(train_df[TARGET_COL])
CLASS_NAMES = le.classes_.tolist()
y_train = le.transform(train_df[TARGET_COL])
y_test  = le.transform(test_df[TARGET_COL])
print(f"Classes: {CLASS_NAMES}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_texts(df, features):
    def row_to_text(row):
        return " | ".join(f"{c}:{v}" for c, v in row.items() if pd.notna(v))
    return df[features].apply(row_to_text, axis=1).tolist()

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    return {
        "accuracy":  accuracy_score(labels, preds),
        "f1":        f1_score(labels, preds, average="weighted", zero_division=0),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall":    recall_score(labels, preds, average="weighted", zero_division=0),
    }

# ============================================================
# ONE EXPERIMENT
# ============================================================
def run_experiment(setting, features, seed):
    print("\n" + "█" * 70)
    print(f" {setting.upper()} | {len(features)} features | seed {seed}")
    print("█" * 70)
    set_seed(seed)

    train_texts = to_texts(train_df, features)
    test_texts  = to_texts(test_df, features)
    print(f"Sample text:\n  {train_texts[0][:300]}...\n")

    # Check token lengths FIRST so truncation can't silently starve the model
    lens = [len(x) for x in tokenizer(train_texts, truncation=False)["input_ids"]]
    max_tok, mean_tok = int(np.max(lens)), float(np.mean(lens))
    trunc_pct = 100.0 * float(np.mean(np.array(lens) > BERT_MAX_LEN))
    max_len = min(BERT_MAX_LEN, max_tok)
    print(f"Token length — mean {mean_tok:.0f} | max {max_tok} | "
          f"truncated at {BERT_MAX_LEN}: {trunc_pct:.2f}% of train rows")
    if trunc_pct > 0:
        print("[WARN] Some rows exceed 512 tokens and WILL be truncated "
              "(features at the end of the string are lost).")

    train_data = Dataset.from_dict({"text": train_texts, "label": y_train.tolist()})
    test_data  = Dataset.from_dict({"text": test_texts,  "label": y_test.tolist()})

    def tok(ex):  # dynamic padding (via collator) -> training time reflects true length
        return tokenizer(ex["text"], truncation=True, max_length=max_len)

    train_data = train_data.map(tok, batched=True)
    test_data  = test_data.map(tok, batched=True)
    cols = ["input_ids", "attention_mask", "label"]
    train_data.set_format("torch", columns=cols)
    test_data.set_format("torch", columns=cols)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(CLASS_NAMES), ignore_mismatched_sizes=True
    ).to(DEVICE)

    # Identical to your original BioMedBERT TrainingArguments
    args = TrainingArguments(
        output_dir=f"./biomedbert_{setting}_s{seed}",
        num_train_epochs=5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=16,
        warmup_steps=50,
        weight_decay=0.01,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        report_to="none",
        seed=seed,
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=train_data, eval_dataset=test_data,
        compute_metrics=compute_metrics,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    )

    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    trainer.train()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    train_sec = time.time() - t0

    pred = trainer.predict(test_data)
    preds = np.argmax(pred.predictions, axis=1)

    print("\n" + "=" * 70)
    print(f"{setting.upper()} RESULTS (seed {seed}) | training time {train_sec/60:.1f} min")
    print("=" * 70)
    print(classification_report(y_test, preds, target_names=CLASS_NAMES, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))

    row = {
        "setting": setting,
        "seed": seed,
        "n_features": len(features),
        "mean_tokens": round(mean_tok, 1),
        "max_tokens": max_tok,
        "truncated_%": round(trunc_pct, 2),
        "train_time_min": round(train_sec / 60, 2),
        "accuracy":  accuracy_score(y_test, preds),
        "f1":        f1_score(y_test, preds, average="weighted", zero_division=0),
        "precision": precision_score(y_test, preds, average="weighted", zero_division=0),
        "recall":    recall_score(y_test, preds, average="weighted", zero_division=0),
    }

    del trainer, model, train_data, test_data
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return row

# ============================================================
# RUN: selected vs all
# ============================================================
rows = []
for seed in SEEDS:
    rows.append(run_experiment("selected_features", SELECTED_FEATURES, seed))
    rows.append(run_experiment("all_features",      ALL_FEATURES,      seed))

res = pd.DataFrame(rows)
res.to_csv("biomedbert_feature_ablation_runs.csv", index=False)

# ============================================================
# COMPARISON
# ============================================================
metric_cols = ["accuracy", "f1", "precision", "recall", "train_time_min"]
summary = res.groupby("setting")[metric_cols + ["n_features", "mean_tokens", "max_tokens", "truncated_%"]] \
             .mean().round(4).loc[["selected_features", "all_features"]]
summary.to_csv("biomedbert_feature_ablation_summary.csv")

print("\n" + "=" * 70)
print(f"COMPARISON — BioMedBERT, mean over seeds {SEEDS}")
print("=" * 70)
print(summary.T)

if len(SEEDS) > 1:
    print("\nStd across seeds:")
    print(res.groupby("setting")[metric_cols].std().round(4).loc[["selected_features", "all_features"]].T)

d = summary.loc["all_features"] - summary.loc["selected_features"]
print("\nΔ (all − selected):")
print(f"  accuracy {d['accuracy']:+.4f} | f1 {d['f1']:+.4f} | "
      f"precision {d['precision']:+.4f} | recall {d['recall']:+.4f}")
print(f"  training time {d['train_time_min']:+.2f} min "
      f"({summary.loc['all_features','train_time_min'] / summary.loc['selected_features','train_time_min']:.2f}x)")
print("\nRead this as: if Δ is within run-to-run noise (or negative), feature selection "
      "is NOT what limits the transformer.")

print("\n✅ Feature ablation completed! CSVs saved: "
      "biomedbert_feature_ablation_runs.csv, biomedbert_feature_ablation_summary.csv")

[OK] transformers
[OK] datasets
[OK] accelerate
[OK] evaluate
Device: cuda
Train: (1776, 23) | Test: (190, 23)
All features: 22 | Selected features: 10
Train Phase distribution:
Phase
Acute        520
Non-MI       434
Chronic      422
Sub-acute    400
Name: count, dtype: int64

Classes: ['Acute', 'Chronic', 'Non-MI', 'Sub-acute']


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]


██████████████████████████████████████████████████████████████████████
 SELECTED_FEATURES | 10 features | seed 42
██████████████████████████████████████████████████████████████████████
Sample text:
  Blood_Glucose_mmolL:12.7 | Diastolic_BP:-0.2367988898819029 | Chest_Pain:1.0 | Weighted_Risk_Score:7.0 | Systolic_BP:-0.3695581856124019 | Heart_Rate_bpm:1.223287249716773 | Weighted_Symptom_Score:7.0 | Serum_Creatinine_mgdL:1.15 | hs_Troponin_I_ngL:1.774542798744036 | Age_Adjusted_Troponin:2.113646...

Token length — mean 136 | max 159 | truncated at 512: 0.00% of train rows


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


ALL_FEATURES RESULTS (seed 42) | training time 4.9 min
              precision    recall  f1-score   support

       Acute       0.92      0.94      0.93        65
     Chronic       0.98      0.98      0.98        52
      Non-MI       0.98      0.98      0.98        55
   Sub-acute       0.76      0.72      0.74        18

    accuracy                           0.94       190
   macro avg       0.91      0.91      0.91       190
weighted avg       0.94      0.94      0.94       190

Confusion Matrix:
 [[61  0  0  4]
 [ 0 51  1  0]
 [ 0  1 54  0]
 [ 5  0  0 13]]

COMPARISON — BioMedBERT, mean over seeds [42]
setting         selected_features  all_features
accuracy                   0.9526        0.9421
f1                         0.9517        0.9416
precision                  0.9515        0.9413
recall                     0.9526        0.9421
train_time_min             3.0800        4.9300
n_features                10.0000       22.0000
mean_tokens              135.7000      242.800